### Оркестрация

#### Установка Airflow

`pip install apache-airflow`

#### Наивный пайплайн

In [ ]:
import random, time

random.seed(1)

def extract():
    # нестабильный источник: в 60% случаев падает (сеть, лимиты API)
    if random.random() < 0.6:
        raise ConnectionError("API timeout")
    return [1, 2, 3, 4, 5]

def transform(rows):
    return [r * 10 for r in rows]

def load(rows):
    print("загружено", rows)

# запуск «как в cron»: один раз, подряд, без защиты
try:
    load(transform(extract()))
except Exception as e:
    print("Пайплайн упал:", repr(e))
    print("Никто не повторил попытку, никто не уведомлён, истории запусков нет.")

# ВОПРОС: чего не хватает этому скрипту? (повторы, состояние, логи,
# зависимости, параллельность, алерты, backfill). Далее добавляем это по частям.

#### Мини-оркестратор

In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from graphlib import TopologicalSorter
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from datetime import datetime
from typing import Callable

# состояния задачи: pending -> running -> success | failed | upstream_failed
class State(str, Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCESS = "success"
    FAILED = "failed"
    UPSTREAM_FAILED = "upstream_failed"

@dataclass
class Task:
    name: str
    fn: Callable
    upstream: list = field(default_factory=list)   # от каких задач зависит
    retries: int = 0                               # сколько повторов разрешено
    retry_delay: float = 0.2                       # пауза, удваивается с каждой попыткой

class MiniDAG:
    def __init__(self, dag_id: str):
        self.dag_id = dag_id
        self.tasks: dict[str, Task] = {}

    # декоратор: регистрирует функцию как задачу графа
    def task(self, after=(), retries=0, retry_delay=0.2):
        def deco(fn):
            t = Task(fn.__name__, fn, [a.name for a in after], retries, retry_delay)
            self.tasks[t.name] = t
            return t
        return deco

    def _log(self, msg):
        print(f"[{datetime.now():%H:%M:%S.%f}"[:-3] + f"] {self.dag_id}: {msg}")

    def _execute(self, t: Task, ctx: dict, results: dict):
        # результаты upstream-задач передаются как аргументы (аналог XCom)
        kwargs = {u: results[u] for u in t.upstream}
        for attempt in range(t.retries + 1):
            try:
                self._log(f"{t.name}: попытка {attempt + 1}/{t.retries + 1}")
                results[t.name] = t.fn(ctx, **kwargs)
                return
            except Exception as e:
                self._log(f"{t.name}: ошибка {e!r}")
                if attempt == t.retries:
                    raise                                   # попытки кончились -> failed
                time.sleep(t.retry_delay * 2 ** attempt)    # экспоненциальная пауза

    def run(self, ctx: dict | None = None, max_workers: int = 4) -> dict:
        ctx = ctx or {}
        graph = {n: set(t.upstream) for n, t in self.tasks.items()}
        sorter = TopologicalSorter(graph)
        sorter.prepare()                      # CycleError, если в графе есть цикл
        state = {n: State.PENDING for n in self.tasks}
        results, running = {}, {}

        with ThreadPoolExecutor(max_workers) as pool:    # «executor»
            while sorter.is_active():
                for name in sorter.get_ready():          # задачи, у которых все upstream завершены
                    t = self.tasks[name]
                    if any(state[u] != State.SUCCESS for u in t.upstream):
                        state[name] = State.UPSTREAM_FAILED      # родитель упал -> не запускаем
                        self._log(f"{name} -> upstream_failed")
                        sorter.done(name)
                        continue
                    state[name] = State.RUNNING
                    running[pool.submit(self._execute, t, ctx, results)] = name
                if not running:
                    continue
                finished, _ = wait(running, return_when=FIRST_COMPLETED)
                for f in finished:
                    name = running.pop(f)
                    state[name] = State.FAILED if f.exception() else State.SUCCESS
                    self._log(f"{name} -> {state[name].value}")
                    sorter.done(name)
        return state

In [ ]:
# Пайплайн: три источника -> transform -> load
#   extract_eu   ─┐
#   extract_us   ─┼─> transform ─> load
#   extract_asia ─┘
attempts = {}      # счётчик, чтобы ошибка была воспроизводимой

dag = MiniDAG("sales_etl")

@dag.task(retries=3)
def extract_eu(ctx):
    time.sleep(0.3); return [1, 2, 3]

@dag.task(retries=3)
def extract_us(ctx):
    attempts["us"] = attempts.get("us", 0) + 1
    if attempts["us"] < 3:                    # первые 2 попытки падают, третья успешна
        raise ConnectionError("API timeout")
    time.sleep(0.3); return [4, 5]

@dag.task(retries=0)
def extract_asia(ctx):
    time.sleep(0.3); return [6]

@dag.task(after=[extract_eu, extract_us, extract_asia])
def transform(ctx, extract_eu, extract_us, extract_asia):
    return [x * 10 for x in extract_eu + extract_us + extract_asia]

@dag.task(after=[transform])
def load(ctx, transform):
    print("   >>> загрузка", transform, "за", ctx.get("ds"))
    return len(transform)

final_state = dag.run({"ds": "2025-01-01"})
print({k: v.value for k, v in final_state.items()})

In [ ]:
# Падение источника навсегда: что происходит с зависимыми задачами?
dag2 = MiniDAG("broken_etl")

@dag2.task(retries=1, retry_delay=0.1)
def extract_a(ctx): raise RuntimeError("источник недоступен")

@dag2.task()
def extract_b(ctx): return [1]

@dag2.task(after=[extract_a, extract_b])
def transform(ctx, extract_a, extract_b): return "не выполнится"

@dag2.task(after=[transform])
def load(ctx, transform): return "не выполнится"

print({k: v.value for k, v in dag2.run().items()})

In [ ]:
# Граф с циклом не имеет порядка выполнения, поэтому он отвергается
from graphlib import CycleError

bad = MiniDAG("cyclic")
a = Task("a", lambda c: 1, upstream=["b"])
b = Task("b", lambda c: 1, upstream=["a"])
bad.tasks = {"a": a, "b": b}
try:
    bad.run()
except CycleError as e:
    print("Отклонено, цикл:", e.args[1])

#### Идемпотентность и backfill

In [ ]:
import json, shutil
from pathlib import Path
from datetime import date, timedelta

OUT = Path("warehouse")
shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir()

def fake_data_for(ds: str):
    # данные детерминированно зависят от логической даты ds (не от now())
    n = int(ds[-2:])
    return [{"day": ds, "amount": 100 + n}]

def load_bad(ds):
    # ПЛОХО: дописывает в конец файла -> каждый повтор создаёт дубли
    with open(OUT / "sales.jsonl", "a") as f:
        for r in fake_data_for(ds):
            f.write(json.dumps(r) + "\n")

def load_good(ds):
    # ХОРОШО: перезаписывает одну партицию целиком и атомарно
    part = OUT / f"ds={ds}"; part.mkdir(exist_ok=True)
    tmp = part / "data.json.tmp"
    tmp.write_text(json.dumps(fake_data_for(ds)))
    tmp.replace(part / "data.json")        # атомарное переименование

# имитируем 2 повтора / ручных перезапуска (всего 3 запуска)
for _ in range(3):
    load_bad("2025-01-01")
    load_good("2025-01-01")

print("bad :", len((OUT / "sales.jsonl").read_text().splitlines()), "строк за один день (дубли!)")
print("good:", len(json.loads((OUT / "ds=2025-01-01" / "data.json").read_text())), "строка за один день")

In [ ]:
def daterange(start: date, end: date):
    d = start
    while d <= end:
        yield d.isoformat()
        d += timedelta(days=1)

for ds in daterange(date(2025, 1, 1), date(2025, 1, 5)):
    load_good(ds)

print(sorted(p.name for p in OUT.glob("ds=*")))

#### Настоящий DAG в Airflow

In [ ]:
# Дожидаемся окончания фоновой установки (если она была запущена в части 0)
import importlib, os

if install_proc is not None:
    print("Ждём установку Airflow...")
    install_proc.wait()
    print("Код возврата pip:", install_proc.returncode, "(0 = успех, иначе см. airflow_install.log)")
importlib.invalidate_caches()

# папка-песочница Airflow; переменные окружения нужно задать ДО import airflow
AIRFLOW_HOME = Path("airflow_home").resolve()
DAGS = AIRFLOW_HOME / "dags"
DAGS.mkdir(parents=True, exist_ok=True)
env = {**os.environ,
       "AIRFLOW_HOME": str(AIRFLOW_HOME),
       "AIRFLOW__CORE__DAGS_FOLDER": str(DAGS),
       "AIRFLOW__CORE__LOAD_EXAMPLES": "False"}
os.environ.update(env)

import airflow
print("Airflow", airflow.__version__)

In [ ]:
%%writefile airflow_home/dags/sales_etl.py
from __future__ import annotations

import json
import random
from datetime import timedelta
from pathlib import Path

import pendulum

try:
    from airflow.sdk import dag, task            # Airflow 3
except ImportError:                              # Airflow 2.x
    from airflow.decorators import dag, task

OUT = Path("/tmp/airflow_course_out")

# В корне файла нет тяжёлого кода и нет datetime.now(): файл парсится постоянно.
@dag(
    dag_id="sales_etl",
    schedule="@daily",                                   # расписание
    start_date=pendulum.datetime(2025, 1, 1, tz="UTC"),
    catchup=False,
    max_active_runs=1,
    default_args={                                       # повторы с экспоненциальной паузой
        "retries": 2,
        "retry_delay": timedelta(seconds=2),
        "retry_exponential_backoff": True,
    },
    tags=["course", "etl"],
)
def sales_etl():

    @task
    def list_regions() -> list[str]:
        return ["eu", "us", "apac"]

    @task
    def extract(region: str, ds=None) -> dict:
        # имитация вызова API за логическую дату ds
        rows = random.Random(f"{region}{ds}").randint(100, 200)
        return {"region": region, "rows": rows}

    @task
    def transform(batch: dict) -> dict:
        batch["rows_clean"] = int(batch["rows"] * 0.95)
        return batch

    @task
    def load(batches: list[dict], ds=None) -> str:
        # идемпотентно: одна партиция на дату, при повторе перезаписывается
        part = OUT / f"ds={ds}"
        part.mkdir(parents=True, exist_ok=True)
        tmp = part / "data.json.tmp"
        tmp.write_text(json.dumps(list(batches)))
        tmp.replace(part / "data.json")
        return str(part / "data.json")                   # передаём ССЫЛКУ, а не данные

    @task(trigger_rule="all_done")                       # выполнится, даже если что-то упало
    def notify(path: str | None = None):
        print(f"Пайплайн завершён. Результат: {path}")

    regions = list_regions()
    extracted = extract.expand(region=regions)           # fan-out: по задаче на регион
    transformed = transform.expand(batch=extracted)      # mapped -> mapped
    path = load(transformed)                             # fan-in: получает список результатов
    notify(path)


sales_etl()

In [ ]:
# Проверка парсинга: то же самое делает DAG processor (и CI в проекте).
# Ошибки импорта и циклы будут видны здесь.
from airflow.models import DagBag

bag = DagBag(dag_folder=str(DAGS), include_examples=False)
print("Ошибки импорта:", bag.import_errors or "нет")
d = bag.dags["sales_etl"]
print("Задачи:", [t.task_id for t in d.tasks])
# ОЖИДАНИЕ: ошибок нет, задач 5: list_regions, extract, transform, load, notify

In [ ]:
# Запуск одного прогона DAG без планировщика (удобно для отладки)
def sh(cmd):
    # вызываем airflow через текущий интерпретатор: работает, даже если его нет в PATH
    if cmd.startswith("airflow "):
        cmd = f"{sys.executable} -m {cmd}"
    r = subprocess.run(cmd, shell=True, env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-2500:])

sh("airflow db migrate")                       # создаёт БД метаданных (SQLite для обучения)
sh("airflow dags test sales_etl 2025-01-03")   # один запуск за логическую дату 2025-01-03